# VoltesFive | HarborCart Phase 3: Visualization Portfolio, Technical Report & Boardroom Pitch
Group: VoltesFive | Section: BSCS-3B | Instructor: Simon Gerard Granil

**Data note:** analysis uses the UCI Online Retail dataset (UK retailer, GBP) as a proxy for the fictional HarborCart. Findings describe this dataset, not Philippine customers.

**Run order (Google Colab):** Phase 1 notebook -> Phase 2 notebook -> this notebook. If the output folders are missing you will be asked to upload `harborcart_phase1_outputs.zip` and `harborcart_phase2_outputs.zip`.

**This notebook generates, from the real Phase 1-2 results:**
1. `figures/` : a 14-figure visualization portfolio, each with a computed insight caption
2. `VoltesFive_Phase3_Technical_Report.pdf` : CRISP-DM technical report with evaluation, ethics and recommendations
3. `VoltesFive_Phase3_Boardroom_Deck.pptx` : 9-slide deck with speaker notes (about 6 minutes)
4. `pitch_script.md` : spoken script and timing

Every number in the report and deck is computed by this notebook. **Read the generated text before submitting** and edit anything your group does not agree with; interpretation is your group's responsibility.

In [ ]:
import subprocess, sys
for mod, pipname in [('reportlab', 'reportlab'), ('pptx', 'python-pptx')]:
    try: __import__(mod)
    except ImportError: subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', pipname])

import json, pickle, zipfile, warnings, textwrap, datetime as dt
from pathlib import Path
import numpy as np, pandas as pd
import matplotlib.pyplot as plt, seaborn as sns, networkx as nx
from sklearn.decomposition import PCA
from sklearn.metrics import roc_curve, roc_auc_score, confusion_matrix
from IPython.display import display

warnings.filterwarnings('ignore')
sns.set_theme(style='whitegrid', context='notebook')
P1, P2, FIG, OUT = Path('harborcart_outputs'), Path('phase2_outputs'), Path('figures'), Path('phase3_outputs')
FIG.mkdir(exist_ok=True); OUT.mkdir(exist_ok=True)

def ensure(folder, marker, zipname):
    if not (folder / marker).exists():
        from google.colab import files
        print(f'Upload {zipname}')
        up = files.upload()
        z = [n for n in up if n.lower().endswith('.zip')]
        assert z, f'Please upload {zipname}'
        folder.mkdir(exist_ok=True)
        with zipfile.ZipFile(z[0]) as zf: zf.extractall(folder)
ensure(P1, 'customer_rfm.csv', 'harborcart_phase1_outputs.zip')
ensure(P2, 'model_artifacts.pkl', 'harborcart_phase2_outputs.zip')

seg = pd.read_csv(P2 / 'customer_segments.csv', dtype={'CustomerID': 'string'}, parse_dates=['FirstPurchase', 'LastPurchase'])
tx = pd.read_csv(P1 / 'rfm_eligible_transactions.csv', dtype={'InvoiceNo': 'string', 'StockCode': 'string', 'CustomerID': 'string'}, parse_dates=['InvoiceDate'])
audit = pd.read_csv(P1 / 'filter_audit.csv'); cond = pd.read_csv(P1 / 'quality_condition_counts.csv', index_col=0).iloc[:, 0]
missing = pd.read_csv(P1 / 'missing_values.csv', index_col=0).iloc[:, 0]
km_grid = pd.read_csv(P2 / 'kmeans_grid.csv'); rules = pd.read_csv(P2 / 'association_rules.csv')
prof = pd.read_csv(P2 / 'segment_profile.csv'); preds = pd.read_csv(P2 / 'test_predictions.csv')
imp = pd.read_csv(P2 / 'model_importance.csv'); val = pd.read_csv(P2 / 'predictive_validation_results.csv')
S = json.load(open(P2 / 'results_summary.json'))
with open(P2 / 'model_artifacts.pkl', 'rb') as f: ART = pickle.load(f)
FEATS = ART['rfm_features']
meta = pd.read_csv(P1 / 'analysis_metadata.csv')
segs_order = prof.sort_values('Revenue_share', ascending=False)['Segment'].tolist()
PAL = dict(zip(segs_order, sns.color_palette('deep', len(segs_order))))
print('Loaded. Customers:', len(seg), '| segments:', segs_order)

## 1. Visualization portfolio (14 figures)
Each figure is saved to `figures/` with an **insight caption computed from the data** (no hard-coded findings).

In [ ]:
CAPS = []   # (file, title, insight)
def keep(fig, fname, title, insight):
    fig.tight_layout(); fig.savefig(FIG / fname, dpi=170, bbox_inches='tight'); plt.show(); plt.close(fig)
    CAPS.append((fname, title, insight)); print('-', title, '|', insight)
gbp = lambda v: f'GBP {v:,.0f}'

# F01 Monthly sales
m = tx.groupby(tx['InvoiceDate'].dt.to_period('M')).agg(sales=('TotalSales', 'sum'), orders=('InvoiceNo', 'nunique'))
m.index = m.index.to_timestamp()
fig, ax = plt.subplots(figsize=(10, 4.5))
ax.plot(m.index, m.sales, marker='o', color='#1f77b4'); ax.set_ylabel('Eligible purchase value (GBP)'); ax.set_title('Monthly eligible purchase value')
ax2 = ax.twinx(); ax2.bar(m.index, m.orders, width=20, alpha=.2, color='gray'); ax2.set_ylabel('Distinct invoices'); ax2.grid(False)
pk = m.sales.idxmax()
keep(fig, 'F01_monthly_sales.png', 'Monthly sales and order counts',
     f'Recorded sales peaked in {pk:%B %Y} at {gbp(m.sales.max())} ({int(m.loc[pk, "orders"]):,} invoices). The final month in the data may be incomplete, so it should not be read as a decline.')

# F02 Top products by revenue
tp = tx.groupby('Description')['TotalSales'].sum().nlargest(10).sort_values()
fig, ax = plt.subplots(figsize=(9, 5)); tp.plot.barh(ax=ax, color='#2a9d8f'); ax.set_xlabel('Eligible sales (GBP)'); ax.set_ylabel('')
ax.set_title('Top 10 products by revenue')
keep(fig, 'F02_top_products.png', 'Top 10 products by revenue',
     f'"{tp.index[-1]}" leads with {gbp(tp.iloc[-1])}, {tp.iloc[-1] / tx.TotalSales.sum():.1%} of all eligible sales; the top 10 together account for {tp.sum() / tx.TotalSales.sum():.1%}.')

# F03 RFM distributions raw vs log
fig, axes = plt.subplots(2, 3, figsize=(13, 6))
for j, c in enumerate(FEATS):
    sns.histplot(seg[c], bins=50, ax=axes[0, j], color='#457b9d'); axes[0, j].set_title(f'{c} (raw)')
    sns.histplot(np.log1p(seg[c]), bins=50, ax=axes[1, j], color='#e76f51'); axes[1, j].set_title(f'{c} (log1p)')
sk = seg[FEATS].skew()
keep(fig, 'F03_rfm_distributions.png', 'RFM distributions before and after log transform',
     f'Raw RFM values are right-skewed (skewness: Recency {sk.Recency:.1f}, Frequency {sk.Frequency:.1f}, Monetary {sk.Monetary:.1f}); log1p makes distance-based clustering more reasonable.')

# F04 Correlation heatmap
cols = FEATS + ['AverageOrderValue', 'CustomerTenure', 'UniqueProducts']
corr = seg[cols].corr(method='spearman')
fig, ax = plt.subplots(figsize=(7.5, 6)); sns.heatmap(corr, annot=True, fmt='.2f', cmap='RdBu_r', vmin=-1, vmax=1, ax=ax); ax.set_title('Spearman correlation')
u = corr.where(np.triu(np.ones(corr.shape), 1).astype(bool)).stack(); a, b = u.abs().idxmax()
keep(fig, 'F04_correlation.png', 'Correlation matrix of customer features',
     f'The strongest relationship is {a} vs {b} (Spearman {corr.loc[a, b]:.2f}). Highly correlated features carry overlapping information.')

# F05 Pareto / Lorenz
s = seg['Monetary'].sort_values(ascending=False).reset_index(drop=True)
cum = s.cumsum() / s.sum(); x = (np.arange(1, len(s) + 1)) / len(s)
top20 = float(cum[int(np.ceil(.2 * len(s))) - 1]); top10 = float(cum[int(np.ceil(.1 * len(s))) - 1])
fig, ax = plt.subplots(figsize=(7, 5)); ax.plot(x * 100, cum * 100, color='#d62828'); ax.plot([0, 100], [0, 100], '--', color='gray')
ax.axvline(20, ls=':', color='k'); ax.set_xlabel('% of customers (highest spenders first)'); ax.set_ylabel('% of eligible sales'); ax.set_title('Customer sales concentration (Pareto)')
keep(fig, 'F05_pareto.png', 'Sales concentration across customers (tests H2)',
     f'The top 10% of customers generate {top10:.1%} and the top 20% generate {top20:.1%} of eligible sales.')

# F06 Model selection
r = km_grid[km_grid.feature_set == 'RFM']; r = r[r.init == 'k-means++']
fig, axes = plt.subplots(1, 3, figsize=(14, 4))
axes[0].plot(r.k, r.inertia, marker='o'); axes[0].set_title('Elbow (inertia)'); axes[0].set_xlabel('k')
axes[1].plot(r.k, r.silhouette, marker='o', color='green'); axes[1].set_title('Silhouette'); axes[1].set_xlabel('k')
axes[2].plot(r.k, r.stability_ARI, marker='o', color='purple'); axes[2].set_title('Stability (ARI across seeds)'); axes[2].set_xlabel('k')
for a_ in axes: a_.axvline(S['kmeans']['k'], ls='--', color='red')
K = S['kmeans']
keep(fig, 'F06_model_selection.png', 'Choosing the number of clusters',
     f'k = {K["k"]} was selected by the pre-set rule (silhouette {K["silhouette"]:.2f}, Davies-Bouldin {K["davies_bouldin"]:.2f}, stability ARI {K["stability_ARI"]:.2f}).')

# F07 PCA scatter
Z = ART['scaler'].transform(np.log1p(seg[FEATS].values)); pca = PCA(2, random_state=42).fit(Z); P = pca.transform(Z)
fig, ax = plt.subplots(figsize=(8, 6))
for sname in segs_order:
    k_ = (seg.Segment == sname).values; ax.scatter(P[k_, 0], P[k_, 1], s=8, alpha=.5, label=sname, color=PAL[sname])
ax.set_xlabel(f'PC1 ({pca.explained_variance_ratio_[0]:.0%})'); ax.set_ylabel(f'PC2 ({pca.explained_variance_ratio_[1]:.0%})'); ax.legend(markerscale=2); ax.set_title('Customer segments (PCA projection)')
keep(fig, 'F07_segments_pca.png', 'Segments projected to 2 dimensions',
     f'The first two components capture {pca.explained_variance_ratio_.sum():.0%} of variance. Visible overlap between groups indicates that boundaries are gradual rather than sharp.')

# F08 Profile heatmap
med = seg.groupby('Segment')[FEATS].median().loc[segs_order]
pct = pd.DataFrame({c: [(seg[c] <= med.loc[sn, c]).mean() * 100 for sn in segs_order] for c in FEATS}, index=segs_order)
fig, ax = plt.subplots(figsize=(8, 0.9 * len(med) + 2))
sns.heatmap(pct, annot=med.round(1), fmt='', cmap='viridis', vmin=0, vmax=100, ax=ax, cbar_kws={'label': 'percentile of all customers'}); ax.set_title('Segment medians (colour = percentile, numbers = actual median)')
keep(fig, 'F08_segment_profile.png', 'What each segment looks like',
     'Colour shows where each segment median falls among all customers (percentile); numbers are actual medians (Recency in days, Monetary in GBP). For Recency, a low percentile is good (bought recently).')

# F09 Customer vs revenue share
share = prof.set_index('Segment').loc[segs_order, ['Customer_share', 'Revenue_share']] * 100
fig, ax = plt.subplots(figsize=(9, 4.5)); share.plot.bar(ax=ax, color=['#8ecae6', '#023047']); ax.set_ylabel('%'); ax.set_xlabel(''); plt.setp(ax.get_xticklabels(), rotation=20, ha='right'); ax.set_title('Share of customers vs share of sales')
top_seg = share.Revenue_share.idxmax()
keep(fig, 'F09_segment_shares.png', 'Customer share vs revenue share by segment',
     f'"{top_seg}" is {share.loc[top_seg, "Customer_share"]:.1f}% of customers but {share.loc[top_seg, "Revenue_share"]:.1f}% of eligible sales.')

# F10 Rules scatter
fig, ax = plt.subplots(figsize=(8, 5.5))
if len(rules):
    sc = ax.scatter(rules.support, rules.confidence, c=rules.lift, cmap='viridis', s=60); plt.colorbar(sc, label='lift')
    ax.set_xlabel('Support'); ax.set_ylabel('Confidence'); ax.set_title('Association rules')
    ins = f'{len(rules)} rules met the thresholds (min support {S["association"]["min_support"]}); the strongest has lift {rules.lift.max():.1f}, i.e. the items co-occur {rules.lift.max():.1f}x more often than if independent.'
else:
    ax.text(.5, .5, 'No rules met thresholds', ha='center'); ins = 'No association rules met the thresholds on this data.'
keep(fig, 'F10_rules_scatter.png', 'Association rules: support, confidence and lift', ins)

# F11 Network
fig, ax = plt.subplots(figsize=(10, 7))
G = nx.DiGraph()
single_rules = rules[~rules.antecedents_txt.str.contains(' + ', regex=False) & ~rules.consequents_txt.str.contains(' + ', regex=False)] if len(rules) else rules
for _, rr in single_rules.head(40).iterrows():
    if True:
        a_, c_ = textwrap.shorten(rr.antecedents_txt, 34), textwrap.shorten(rr.consequents_txt, 34)
        if a_ != c_: G.add_edge(a_, c_, lift=rr.lift)
if G.number_of_nodes():
    pos = nx.spring_layout(G, seed=42, k=1.1); dc = nx.degree_centrality(G)
    nx.draw_networkx_nodes(G, pos, node_size=[300 + 2500 * dc[n_] for n_ in G], node_color='#ffb703', ax=ax)
    nx.draw_networkx_edges(G, pos, arrows=True, alpha=.5, ax=ax); nx.draw_networkx_labels(G, pos, font_size=7, ax=ax)
    hub = max(dc, key=dc.get); ins = f'The rule network has {G.number_of_nodes()} products and {G.number_of_edges()} links; "{hub}" is the most connected product.'
else:
    ins = 'No single-item rules were available to draw a network.'
ax.set_axis_off(); ax.set_title('Product association network (top single-item rules by lift)')
keep(fig, 'F11_rule_network.png', 'Product association network', ins)

# F12 Confusion matrix
cm = confusion_matrix(preds.y_true, preds.y_pred)
fig, ax = plt.subplots(figsize=(5, 4.2)); sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', cbar=False, ax=ax, xticklabels=['No', 'Yes'], yticklabels=['No', 'Yes'])
ax.set_xlabel('Predicted: buys in next 90 days'); ax.set_ylabel('Actual'); ax.set_title('Confusion matrix (test set)')
T, B = S['predictive']['test'], S['predictive']['baseline_test']
keep(fig, 'F12_confusion_matrix.png', 'Predictive model: confusion matrix',
     f'On the held-out test set accuracy is {T["accuracy"]:.1%} (baseline {B["accuracy"]:.1%}); precision {T["precision"]:.1%}, recall {T["recall"]:.1%}.')

# F13 ROC
fpr, tpr, _ = roc_curve(preds.y_true, preds.y_prob); auc = roc_auc_score(preds.y_true, preds.y_prob)
fig, ax = plt.subplots(figsize=(5.5, 5)); ax.plot(fpr, tpr, label=f'{S["predictive"]["selected_model"]} (AUC {auc:.2f})'); ax.plot([0, 1], [0, 1], '--', color='gray', label='Chance (AUC 0.50)')
ax.set_xlabel('False positive rate'); ax.set_ylabel('True positive rate'); ax.legend(loc='lower right'); ax.set_title('ROC curve (test set)')
keep(fig, 'F13_roc_curve.png', 'Predictive model: ROC curve',
     f'Test ROC-AUC is {auc:.2f} versus 0.50 for chance, showing how well the model ranks likely repeat buyers above others.')

# F14 Importance
fig, ax = plt.subplots(figsize=(7, 4)); ii = imp.sort_values('importance_mean')
ax.barh(ii.feature, ii.importance_mean, xerr=ii.importance_std, color='#6a4c93'); ax.set_xlabel('Drop in ROC-AUC when feature is shuffled'); ax.set_title('Permutation feature importance (test)')
keep(fig, 'F14_feature_importance.png', 'What drives the repeat-purchase prediction',
     f'"{imp.iloc[0].feature}" is the most influential feature; this is predictive association, not proof of cause.')

## 2. Hypothesis evaluation (computed)
Verdict rules are heuristics stated here so they are transparent; your group may adjust and justify them.

In [ ]:
q1, med_, q3 = seg.Monetary.quantile([.25, .5, .75]); f1_, f3_ = seg.Frequency.quantile([.25, .75])
cv = seg.Monetary.std() / seg.Monetary.mean()
h1 = ('Supported', f'Spending varies widely: middle-50% range is {gbp(q1)} to {gbp(q3)} (coefficient of variation {cv:.2f}); purchase frequency middle-50% range is {f1_:.0f} to {f3_:.0f} invoices.')
h2 = ('Supported' if top20 >= .5 else 'Not supported', f'Top 20% of customers = {top20:.1%} of eligible sales (rule: supported if >= 50%).')
sil, ari = K['silhouette'], K['stability_ARI']
v3 = 'Supported' if (sil >= .25 and ari >= .7) else ('Partially supported' if sil >= .15 else 'Not supported')
h3 = (v3, f'k={K["k"]}: silhouette {sil:.2f}, stability ARI {ari:.2f} (rule: supported if silhouette >= 0.25 and ARI >= 0.7; partial if silhouette >= 0.15). Segments were also checked for business interpretability in the profile table.')
HYP = [('H1', 'Customers differ in purchase frequency and total spending.') + h1,
       ('H2', 'A smaller group contributes a substantial portion of recorded sales.') + h2,
       ('H3', 'RFM features support distinguishable, interpretable groups.') + h3]
display(pd.DataFrame(HYP, columns=['Hypothesis', 'Statement', 'Verdict', 'Evidence']))

## 3. Recommendations by segment
Rule-based playbook keyed on the segment label from Phase 2. These are **proposals to test** (e.g., via A/B tests), not verified effects.

In [ ]:
PLAY = {
 'Loyal High-Value': ('Protect and reward', 'Loyalty tiers, early access to new products, personal service. Track churn signals closely.'),
 'Active Regulars': ('Grow basket size', 'Bundle offers and cross-sell using the association rules; free-shipping thresholds just above average order value.'),
 'Recent Big Spenders': ('Convert to repeat', 'Follow-up offer after the large order and a reason to return (restock reminders).'),
 'Recent Low-Frequency': ('Nurture new/occasional buyers', 'Welcome series, second-purchase incentive, product recommendations from popular pairs.'),
 'At-Risk High-Value': ('Win back quickly', 'Priority re-engagement (personal outreach, tailored offer) because these customers were valuable but have gone quiet.'),
 'Lapsed Low-Value': ('Low-cost reactivation', 'Automated email/SMS win-back with a capped budget; stop spending if there is no response.'),
}
def base(n): return n.split(' (Tier')[0]
rec = []
for sname in segs_order:
    row = prof.set_index('Segment').loc[sname]
    goal, act = PLAY.get(base(sname), ('Review manually', 'Inspect this segment before choosing an action.'))
    rec.append([sname, f'{int(row.Customers):,} ({row.Customer_share:.1%})', f'{row.Revenue_share:.1%}',
                f'{row.Recency_median:.0f} d / {row.Frequency_median:.0f} / {gbp(row.Monetary_median)}', goal, act])
REC = pd.DataFrame(rec, columns=['Segment', 'Customers', 'Revenue share', 'Median R / F / M', 'Goal', 'Suggested action'])
display(REC)

## 4. Build the PDF technical report

In [ ]:
from reportlab.lib.pagesizes import letter
from reportlab.lib import colors
from reportlab.lib.units import inch
from reportlab.lib.styles import getSampleStyleSheet, ParagraphStyle
from reportlab.platypus import SimpleDocTemplate, Paragraph, Spacer, Table, TableStyle, Image, PageBreak, KeepTogether
from PIL import Image as PILImage
from xml.sax.saxutils import escape as esc

st = getSampleStyleSheet()
B_ = ParagraphStyle('B', parent=st['BodyText'], fontSize=10, leading=14, spaceAfter=6)
SM = ParagraphStyle('SM', parent=B_, fontSize=8, leading=10, spaceAfter=0)
CAP = ParagraphStyle('CAP', parent=B_, fontSize=9, leading=12, textColor=colors.HexColor('#333333'), spaceAfter=10)
H1 = ParagraphStyle('H1', parent=st['Heading1'], fontSize=15, spaceBefore=10, spaceAfter=6, keepWithNext=1)
H2 = ParagraphStyle('H2', parent=st['Heading2'], fontSize=12, spaceBefore=8, spaceAfter=4, keepWithNext=1)
BUL = ParagraphStyle('BUL', parent=B_, leftIndent=14, bulletIndent=2, spaceAfter=3)
def bl(items): return [Paragraph(i, BUL, bulletText='\u2022') for i in items]
P = lambda t, s=B_: Paragraph(t, s)

def tbl(df, widths=None):
    data = [[P(f'<b>{esc(str(c).replace(chr(95), chr(32)))}</b>', SM) for c in df.columns]] + [[P(esc(str(v)), SM) for v in r] for r in df.values.tolist()]
    t = Table(data, colWidths=widths, repeatRows=1)
    t.setStyle(TableStyle([('BACKGROUND', (0, 0), (-1, 0), colors.HexColor('#dfe7f2')), ('GRID', (0, 0), (-1, -1), .4, colors.grey), ('VALIGN', (0, 0), (-1, -1), 'TOP')]))
    return t

def fig_(fname, width=6.0):
    cap = next(c for c in CAPS if c[0] == fname); w, h = PILImage.open(FIG / fname).size
    n_ = fname.split('_')[0]
    return KeepTogether([Image(str(FIG / fname), width=width * inch, height=width * inch * h / w),
                         P(f'<b>Figure {int(n_[1:])}. {esc(cap[1])}.</b> {esc(cap[2])}', CAP)])

rb = lambda v: f'{v:.3f}'
n_raw = int(audit['removed_at_step'].sum() + audit['remaining'].iloc[-1])
story = []
story += [P('HarborCart Customer Segmentation and Sales Analysis', st['Title']),
          P('Phase 3: Technical Report and Recommendations', st['Heading2']),
          P('Group: VoltesFive | Section: BSCS-3B | Course: CSE315 Data Science | Instructor: Simon Gerard Granil'),
          P('Members: Joseph Ramos, Christopher Ladiero, Aaron Falsario, Romualnico Ignacio, Darwin James De Leon'),
          P(f'Generated: {dt.date.today():%B %d, %Y}'),
          P('<b>Data note.</b> HarborCart is fictional. The analysis uses the public UCI Online Retail dataset (Chen, 2015; CC BY 4.0), a UK retailer with prices in GBP, as a proxy. Results describe that dataset and are not evidence of Philippine market behavior.')]

# Executive summary
seg_lines = [f'<b>{esc(r.Segment)}</b>: {r.Customer_share:.1%} of customers, {r.Revenue_share:.1%} of sales.' for r in prof.set_index('Segment').loc[segs_order].reset_index().itertuples()]
story += [P('1. Executive summary', H1),
          P(f'We analysed {S["n_lines"]:,} eligible purchase lines from {S["n_customers"]:,} identified customers. The top 20% of customers account for {top20:.1%} of eligible sales. '
            f'k-Means on log-scaled RFM produced {K["k"]} segments (silhouette {K["silhouette"]:.2f}, stability ARI {K["stability_ARI"]:.2f}):'),
          *bl(seg_lines),
          P(f'The selected model ({S["predictive"]["selected_model"]}) predicts whether a customer buys again in the last {S["predictive"]["horizon_days"]} days with test ROC-AUC {T["roc_auc"]:.2f} (baseline {B["roc_auc"]:.2f}). '
            f'Hypotheses: H1 {h1[0].lower()}, H2 {h2[0].lower()}, H3 {h3[0].lower()}.')]

# Business understanding
story += [P('2. Business understanding', H1),
          P('HarborCart E-Commerce, Inc. (fictional) lacks a structured way to group customers by purchasing behavior. <b>Objective:</b> apply CRISP-DM to prepare RFM features, segment customers, explore co-purchase patterns, predict repeat purchasing and translate findings into testable recommendations. '
            '<b>Questions:</b> purchase frequency, spending variation, top products and periods, whether RFM supports meaningful groups, and how segments could guide marketing and retention.')]

# Data
aud = audit.rename(columns={'rule': 'Exclusion rule (sequential)', 'removed_at_step': 'Rows removed', 'remaining': 'Rows remaining'})
story += [P('3. Data understanding and preparation', H1),
          P(f'The original file had {n_raw:,} rows. Missing values: ' + ', '.join(f'{k} {int(v):,}' for k, v in missing.items() if v > 0) + '. '
            'Exact duplicates were counted but retained; cancellations were separated; rows without CustomerID were excluded from customer-level RFM; no IDs were imputed; returns were not netted against purchases.'),
          tbl(aud, [2.6 * inch, 1.5 * inch, 1.5 * inch]), Spacer(1, 8),
          P(f'RFM reference date: {meta.loc[0, "reference_date"]} (day after the last eligible invoice); frequency counts distinct invoices; monetary value is in GBP.')]

# EDA
story += [P('4. Exploratory analysis', H1), fig_('F01_monthly_sales.png'), fig_('F02_top_products.png', 5.4), fig_('F03_rfm_distributions.png'), fig_('F04_correlation.png', 4.4), fig_('F05_pareto.png', 4.4)]

# Modeling
kv = km_grid[(km_grid.feature_set == 'RFM') & (km_grid.init == 'k-means++')][['k', 'silhouette', 'davies_bouldin', 'calinski_harabasz', 'min_cluster_share', 'stability_ARI']].round(3)
pt = prof.copy(); pt['Customers'] = pt.Customers.map('{:,}'.format); pt['Customer_share'] = pt.Customer_share.map('{:.1%}'.format); pt['Revenue_share'] = pt.Revenue_share.map('{:.1%}'.format)
pt = pt[['Segment', 'Customers', 'Customer_share', 'Revenue_share', 'Recency_median', 'Frequency_median', 'Monetary_median']].round(1)
story += [P('5. Modeling and evaluation', H1), P('5.1 Customer segmentation (k-Means)', H2),
          P(f'Features: log1p(Recency, Frequency, Monetary), standardized. {int((km_grid.shape[0]))} configurations were tried (k, initialization, feature set; see Model_Experiment_Log.xlsx). '
            'Selection rule fixed in advance: k >= 3, smallest cluster >= 3%, stability ARI >= 0.6, highest silhouette.'),
          tbl(kv), Spacer(1, 6), fig_('F06_model_selection.png'), tbl(pt), Spacer(1, 6), fig_('F07_segments_pca.png', 5.0), fig_('F08_segment_profile.png', 5.2), fig_('F09_segment_shares.png', 5.4),
          P('5.2 Association rules (mlxtend Apriori)', H2),
          P(f'{S["association"]["baskets"]:,} baskets (invoices with >= 2 of the 150 most frequent products), min support {S["association"]["min_support"]}. Rules kept: lift >= 1.2 and confidence >= 0.3.')]
if len(rules):
    tr = rules.head(8).copy(); tr[['support', 'confidence', 'lift']] = tr[['support', 'confidence', 'lift']].round(3)
    story += [tbl(tr, [1.9 * inch, 1.9 * inch, .8 * inch, .8 * inch, .6 * inch]), Spacer(1, 6)]
story += [fig_('F10_rules_scatter.png', 4.8), fig_('F11_rule_network.png', 5.6),
          P('5.3 Predictive model: repeat purchase in the final 90 days', H2),
          P(f'Features come only from purchases before {S["predictive"]["cutoff"]} (avoiding leakage); target = any purchase afterwards. {S["predictive"]["n_customers"]:,} customers, repeat rate {S["predictive"]["positive_rate"]:.1%}. '
            f'Split {S["predictive"]["split_sizes"]} (train/validation/test, stratified). Logistic Regression, Decision Tree and MLP variants were tuned on validation ROC-AUC; the test set was used once.')]
vt = val.round(3); story += [tbl(vt.head(8)), Spacer(1, 6)]
tm = pd.DataFrame({'Selected model': T, 'Baseline': B}).T.round(3).reset_index().rename(columns={'index': 'Test results'})
story += [tbl(tm), Spacer(1, 6), fig_('F12_confusion_matrix.png', 3.6), fig_('F13_roc_curve.png', 3.8), fig_('F14_feature_importance.png', 5.0)]

# Hypotheses
story += [P('6. Hypothesis evaluation', H1), tbl(pd.DataFrame(HYP, columns=['ID', 'Hypothesis', 'Verdict', 'Evidence']), [.4 * inch, 2.0 * inch, 1.0 * inch, 3.0 * inch])]

# Recommendations
story += [P('7. Business recommendations', H1), P('These proposals follow from the segment profiles above. Benefits are unverified until tested (for example with an A/B test on a small share of each segment).'),
          tbl(REC, [1.1 * inch, .9 * inch, .6 * inch, 1.2 * inch, 1.0 * inch, 2.2 * inch])]

# Ethics
story += [P('8. Ethical, privacy and security considerations', H1),
          *bl(['<b>Data origin:</b> public, anonymized proxy data (CC BY 4.0, cited). No real HarborCart or Philippine customer records or credentials were used.', '<b>Privacy law:</b> a real deployment would fall under the Philippine Data Privacy Act of 2012 (RA 10173): collect only needed data, state purpose, obtain consent where required, and honor access/erasure rights.', '<b>Access control:</b> store data, scripts and outputs in the approved repository with role-based access; share segment-level summaries rather than individual rows where possible.', '<b>Fairness:</b> segments must not be used to deny service or discriminate; spending-based labels such as "Lapsed Low-Value" should not drive differential treatment of people beyond marketing contact.', '<b>Transparency:</b> customers should not be profiled in ways they would not reasonably expect; keep documentation of model purpose and limits.', '<b>Misuse risk:</b> a repeat-purchase score could drive aggressive targeting; apply contact-frequency caps and opt-out.'])]

# Limitations
story += [P('9. Limitations', H1),
          *bl(['UK retailer, 2010-2011, GBP: not representative of Philippine e-commerce or of today.', 'Only customers with a CustomerID and positive purchases are included; cancellations and returns are not netted, so monetary value is gross purchases.', 'Roughly one year of data limits seasonality and churn analysis; the final month may be partial.', 'Segment names are interpretations of median values, and RFM clusters are partly an arbitrary partition of a continuum.', 'The predictive target uses a single 90-day window and a random customer split; temporal robustness is untested.', 'Association rules use only the 150 most frequent products and are not causal.']),
          P('10. CRISP-DM deployment note', H1),
          P('This project delivers reproducible notebooks, a saved model artifact (model_artifacts.pkl), and recommendations. It does not claim a production deployment; next steps would be a pilot test and monitoring of segment drift.'),
          P('References', H1), P('Chen, D. (2015). Online Retail [Dataset]. UCI Machine Learning Repository. https://doi.org/10.24432/C5BW33')]

REPORT = OUT / 'VoltesFive_Phase3_Technical_Report.pdf'
def footer(c, d):
    c.saveState(); c.setFont('Helvetica', 8); c.drawString(.75 * inch, .5 * inch, 'VoltesFive | HarborCart Phase 3 | proxy data (UCI Online Retail)'); c.drawRightString(7.75 * inch, .5 * inch, f'Page {d.page}'); c.restoreState()
SimpleDocTemplate(str(REPORT), pagesize=letter, leftMargin=.75 * inch, rightMargin=.75 * inch, topMargin=.7 * inch, bottomMargin=.8 * inch,
                  title='HarborCart Phase 3 Technical Report', author='VoltesFive').build(story, onFirstPage=footer, onLaterPages=footer)
print('Saved', REPORT)

## 5. Build the boardroom deck and pitch script
Nine slides, about 40 seconds each = 5-7 minutes. Speaker notes are included in the PPTX and in `pitch_script.md`.

In [ ]:
from pptx import Presentation
from pptx.util import Inches, Pt
from pptx.dml.color import RGBColor

NAVY, TEAL, GREY = RGBColor(0x0B, 0x25, 0x45), RGBColor(0x13, 0x7C, 0x8B), RGBColor(0x44, 0x44, 0x44)
prs = Presentation(); prs.slide_width, prs.slide_height = Inches(13.333), Inches(7.5)
blank = prs.slide_layouts[6]
SCRIPT = []

def text(slide, x, y, w, h, s, size=18, bold=False, color=GREY):
    tb = slide.shapes.add_textbox(Inches(x), Inches(y), Inches(w), Inches(h)); tf = tb.text_frame; tf.word_wrap = True
    for i, line in enumerate(s if isinstance(s, list) else [s]):
        p = tf.paragraphs[0] if i == 0 else tf.add_paragraph()
        p.text = line; p.font.size = Pt(size); p.font.bold = bold; p.font.color.rgb = color; p.space_after = Pt(14)
    return tb

def slide(title, bullets=None, img=None, notes='', img_w=6.6, bullets_w=5.6):
    s = prs.slides.add_slide(blank)
    bar = s.shapes.add_shape(1, 0, 0, prs.slide_width, Inches(1.0)); bar.fill.solid(); bar.fill.fore_color.rgb = NAVY; bar.line.fill.background()
    text(s, .5, .18, 12.3, .7, title, 28, True, RGBColor(255, 255, 255))
    if bullets: text(s, .5, 1.3, bullets_w if img else 12.3, 5.5, ['- ' + b for b in bullets], 22)
    if img:
        w, h = PILImage.open(FIG / img).size; iw = img_w; ih = iw * h / w
        if ih > 5.6: ih = 5.6; iw = ih * w / h
        s.shapes.add_picture(str(FIG / img), Inches(13.333 - iw - .4), Inches(1.3), Inches(iw), Inches(ih))
    text(s, .5, 7.0, 12, .4, 'VoltesFive | HarborCart (fictional) | proxy data: UCI Online Retail, UK, GBP', 10)
    s.notes_slide.notes_text_frame.text = notes; SCRIPT.append((title, notes))

segr = prof.set_index('Segment').loc[segs_order]
tops = segr.iloc[0]
slide('HarborCart: Who are our customers and what should we do?',
      ['Customer segmentation and sales analysis using CRISP-DM', 'Group VoltesFive | BSCS-3B | CSE315 Data Science',
       'Proxy data: UCI Online Retail (UK retailer, GBP), not real HarborCart data'],
      notes='Good day. We are VoltesFive. HarborCart is a fictional Philippine marketplace. We used a public UK retail dataset as a stand-in to show how customer data can guide marketing and retention decisions. All findings describe that dataset.')
slide('The business problem',
      ['HarborCart has no structured way to group customers by behavior', 'Goal: segment customers on Recency, Frequency, Monetary value (RFM)', 'Then turn segments into testable marketing actions',
       f'Data: {S["n_lines"]:,} purchase lines, {S["n_customers"]:,} customers'],
      notes='Without segments, every customer gets the same message. We asked three questions: how do customers differ, which groups matter most, and what should HarborCart do for each.')
slide('Our approach: CRISP-DM end to end',
      ['Prepare: removed cancellations, rows without customer ID, non-positive values (every exclusion counted)', 'Describe: k-Means segments, association rules, rule network',
       'Predict: will a customer buy again in the next 90 days?', f'{S["n_experiments"]} experiments logged; test set used once'],
      notes='Data preparation followed the audit trail in our report. We logged every experiment, chose the number of clusters with a rule fixed in advance, and avoided leakage in the prediction task by building features only from before the cutoff date.')
slide('A few customers drive most of the sales', [f'Top 10% of customers: {top10:.0%} of sales', f'Top 20% of customers: {top20:.0%} of sales', 'Spending varies enormously across customers', 'Losing a top customer costs far more than losing an average one'],
      img='F05_pareto.png', img_w=5.6,
      notes=f'Sales are concentrated. The top twenty percent of customers account for {top20:.0%} of recorded sales. This is why protecting high-value customers is a priority.')
slide(f'{K["k"]} customer segments', [f'{r.Index}: {r.Customer_share:.0%} of customers, {r.Revenue_share:.0%} of sales' for r in segr.itertuples()] + [f'Silhouette {K["silhouette"]:.2f}; stability ARI {K["stability_ARI"]:.2f}'],
      img='F09_segment_shares.png', img_w=6.4, bullets_w=6.0,
      notes=f'k-Means on log-scaled RFM gave {K["k"]} groups. The biggest revenue segment, {segr.index[0]}, is {tops.Customer_share:.0%} of customers but {tops.Revenue_share:.0%} of sales. Cluster quality is {"good" if K["silhouette"] >= .5 else "moderate" if K["silhouette"] >= .25 else "weak"} by silhouette, so we treat segments as useful guides, not hard boundaries.')
slide('What each segment looks like', ['Colour: percentile among all customers (low Recency = recent)', 'Numbers: actual median Recency (days), Frequency, Monetary (GBP)', 'Names are our interpretation of these medians'],
      img='F08_segment_profile.png', img_w=6.8, bullets_w=5.2,
      notes='This heatmap shows how each group behaves. Recency is days since last purchase, so lower is better. Names such as Lapsed or Loyal are labels we assigned from the medians.')
if len(rules):
    best_rule = rules.iloc[0]
    slide('Products bought together', [f'{len(rules)} strong rules (lift >= 1.2, confidence >= 0.3)', f'Top rule: {textwrap.shorten(best_rule.antecedents_txt, 40)} -> {textwrap.shorten(best_rule.consequents_txt, 40)}', f'Lift {best_rule.lift:.1f}, confidence {best_rule.confidence:.0%}', 'Use for bundles and recommendations'],
          img='F11_rule_network.png', img_w=6.6, bullets_w=5.6,
          notes=f'Market-basket analysis found {len(rules)} strong product pairs. The strongest has lift {best_rule.lift:.1f}, meaning the items appear together that many times more often than chance. These are ideas for bundles and cross-sell, not proof of cause.')
else:
    slide('Products bought together', ['No rules met the thresholds on this data'], notes='No association rules met our thresholds, so we do not recommend product bundles from this analysis.')
slide('Predicting repeat purchases', [f'Model: {S["predictive"]["selected_model"]}', f'Test ROC-AUC {T["roc_auc"]:.2f} (baseline {B["roc_auc"]:.2f})', f'Precision {T["precision"]:.0%}, recall {T["recall"]:.0%}', f'Top driver: {imp.iloc[0].feature}', 'Predictive, not causal'],
      img='F13_roc_curve.png', img_w=4.8, bullets_w=6.8,
      notes=f'We predict whether a customer buys again in the final ninety days. On a held-out test set the model reaches an AUC of {T["roc_auc"]:.2f}, compared with 0.50 for guessing. The most influential feature is {imp.iloc[0].feature}.')
rec_lines = [f'{r.Segment}: {r.Goal}' for r in REC.itertuples()]
slide('Recommendations and next steps', rec_lines[:5] + ['Pilot with A/B tests; respect privacy (RA 10173); validate on real HarborCart data'],
      notes='Each segment gets a different goal. Benefits are unverified, so we recommend small A/B tests first. Limits: the data is a UK retailer from 2010 to 2011, so real HarborCart data must be used before any decision. Thank you.')

DECK = OUT / 'VoltesFive_Phase3_Boardroom_Deck.pptx'; prs.save(DECK)
with open(OUT / 'pitch_script.md', 'w') as f:
    f.write('# Boardroom pitch script (target 5-7 minutes, about 40 s per slide)\n\n')
    for i, (t, n_) in enumerate(SCRIPT, 1): f.write(f'## Slide {i}: {t}\n{n_}\n\n')
words = sum(len(n_.split()) for _, n_ in SCRIPT)
print('Saved', DECK, f'| script about {words} words = ~{words / 140:.1f} min at 140 wpm')

## 6. Package and download

In [ ]:
import shutil
shutil.copytree(FIG, OUT / 'figures', dirs_exist_ok=True)
arc = Path('harborcart_phase3_outputs.zip')
with zipfile.ZipFile(arc, 'w', zipfile.ZIP_DEFLATED) as z:
    for p in sorted(OUT.rglob('*')):
        if p.is_file(): z.write(p, arcname=str(p.relative_to(OUT)))
print('Saved', arc)
try:
    from google.colab import files; files.download(str(arc))
except ImportError:
    pass